In [2]:
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, average_precision_score, precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

X_train_smote = pd.read_parquet('../member2_outputs/X_train_smote.parquet')
y_train_smote = pd.read_parquet('../member2_outputs/y_train_smote.parquet').squeeze()
X_test = pd.read_parquet('../data/processed/X_test.parquet')
y_test = pd.read_parquet('../data/processed/y_test.parquet').squeeze()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}

trained_models = {}
results = []

for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    trained_models[name] = model
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.sort_values(by="PR-AUC", ascending=False))

best_model_name = results_df['PR-AUC'].idxmax()
joblib.dump(trained_models[best_model_name], f'../artifacts/{best_model_name.replace(" ", "_").lower()}_fraud_model.joblib')
print(f"Model ({best_model_name}) successfully saved to artifacts folder for Member 4!")

,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
Model,,,,,
XGBoost,0.808511,0.800000,0.804233,0.971800,0.820730
Random Forest,0.934211,0.747368,0.830409,0.954592,0.819639
Logistic Regression,0.397906,0.800000,0.531469,0.965719,0.689322


Model (XGBoost) successfully saved to artifacts folder for Member 4!
